# Notebook 11 – Feature Engineering y Modelado Avanzado

**Objetivo:** Mejorar los modelos baseline del notebook 10 mediante:
1. Ingeniería de características: interacciones, índices compuestos satelitales, tendencias temporales
2. Optimización del umbral de clasificación (curva ROC, maximizar Recall ≥ 0.70)
3. Validación leave-one-department-out (LOO)
4. Calibración del trigger del seguro (curva de basis risk)
5. Ensamble de stacking

**Datasets:** `dataset_modelado_anual_limpio.csv` (36 obs × 81 vars)

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.ensemble import (
    ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestRegressor,
    StackingRegressor
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet, HuberRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, auc, precision_recall_curve, f1_score
)
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost no disponible')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR = Path(r'C:\Users\DiDi\Downloads\Proyecto Analytics')
DATA_DIR = BASE_DIR / 'BASE_DE_DATOS' / 'FINALES'
OUT_DIR  = BASE_DIR / 'MODELOS' / 'resultados_11_avanzado'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END_YEAR = 2020
TARGET         = 'perdida_rendimiento_anual_pct'
TARGET_CLF     = 'evento_perdida_anual'
LOSS_THRESHOLD = -15.0   # % umbral del seguro
RANDOM_STATE   = 42

print('Setup completo')
print('Output dir:', OUT_DIR)

## 1. Carga de datos

In [ ]:
df_raw = pd.read_csv(DATA_DIR / 'dataset_modelado_anual_limpio.csv', sep=';')
print(f'Shape: {df_raw.shape}')
print('Años:', sorted(df_raw['anio'].unique()))
print('Departamentos:', df_raw['departamento'].unique())
print('\nDistribución del target:')
print(df_raw[TARGET].describe().round(2))
print(f'\nEventos de pérdida (≤ {LOSS_THRESHOLD}%): {df_raw[TARGET_CLF].sum()}/{len(df_raw)}')

## 2. Ingeniería de características

In [ ]:
df = df_raw.copy()

eps = 1e-6  # evitar división por cero

# ── 2.1 Interacciones clima × clima ─────────────────────────────────────────
df['temp_x_def_annual']  = df['temp_aire_C_annual_mean']  * df['def_annual_mean']
df['temp_x_def_cosecha'] = df['temp_aire_C_cosecha_mean'] * df['def_cosecha_mean']

df['precip_x_ndvi_anom_annual']  = df['precipitation_annual_sum'] * df['NDVI_anomalia_pct_annual_mean']
df['precip_x_ndvi_anom_cosecha'] = df['precipitation_cosecha_sum'] * df['NDVI_anomalia_pct_cosecha_mean']

df['GDD_x_gpp_anom_annual']  = df['GDD_cafe_annual_mean']  * df['Gpp_anomalia_pct_annual_mean']
df['GDD_x_gpp_anom_cosecha'] = df['GDD_cafe_cosecha_mean'] * df['Gpp_anomalia_pct_cosecha_mean']

# ── 2.2 Índices compuestos satelitales ──────────────────────────────────────
df['gpp_ndvi_ratio_annual']  = df['Gpp_annual_mean']  / (df['NDVI_annual_mean']  + eps)
df['gpp_ndvi_ratio_cosecha'] = df['Gpp_cosecha_mean'] / (df['NDVI_cosecha_mean'] + eps)

df['stress_hidrico_annual']  = df['def_annual_mean']  / (df['precipitation_annual_sum']  + eps)
df['stress_hidrico_cosecha'] = df['def_cosecha_mean'] / (df['precipitation_cosecha_sum'] + eps)

df['balance_hidrico_annual']  = df['aet_annual_mean']  - df['pet_annual_mean']
df['balance_hidrico_cosecha'] = df['aet_cosecha_mean'] - df['pet_cosecha_mean']

df['amplitud_termica_annual']  = df['LST_Day_1km_annual_mean']  - df['LST_Night_1km_annual_mean']
df['amplitud_termica_cosecha'] = df['LST_Day_1km_cosecha_mean'] - df['LST_Night_1km_cosecha_mean']

# ── 2.3 Tendencias temporales (lag 1 y 2 años por departamento) ─────────────
df = df.sort_values(['departamento', 'anio']).reset_index(drop=True)

for var in ['NDVI_anomalia_pct_annual_mean', 'Gpp_anomalia_pct_annual_mean',
            'def_annual_mean', 'temp_aire_C_annual_mean', 'precipitation_annual_sum']:
    short = var.replace('_annual_mean', '').replace('_annual_sum', '')
    df[f'{short}_lag1']  = df.groupby('departamento')[var].shift(1)
    df[f'{short}_lag2']  = df.groupby('departamento')[var].shift(2)
    df[f'{short}_delta'] = df[var] - df[f'{short}_lag1']

df['anio_relativo'] = df['anio'] - df['anio'].min()

# ── 2.4 Índice agregado de estrés ─────────────────────────────────────────
stress_vars = [
    'NDVI_anomalia_pct_annual_mean',
    'Gpp_anomalia_pct_annual_mean',
    'precipitation_anomalia_pct_annual_mean',
]
_has_data = df[stress_vars].notna().all(axis=1)
for v in stress_vars:
    _mu  = df.loc[_has_data, v].mean()
    _std = df.loc[_has_data, v].std() + eps
    df[f'{v}_z'] = (df[v] - _mu) / _std

z_cols = [f'{v}_z' for v in stress_vars]
df['indice_estres_compuesto'] = df[z_cols].mean(axis=1)

new_features = [
    'temp_x_def_annual', 'temp_x_def_cosecha',
    'precip_x_ndvi_anom_annual', 'precip_x_ndvi_anom_cosecha',
    'GDD_x_gpp_anom_annual', 'GDD_x_gpp_anom_cosecha',
    'gpp_ndvi_ratio_annual', 'gpp_ndvi_ratio_cosecha',
    'stress_hidrico_annual', 'stress_hidrico_cosecha',
    'balance_hidrico_annual', 'balance_hidrico_cosecha',
    'amplitud_termica_annual', 'amplitud_termica_cosecha',
    'NDVI_anomalia_pct_lag1', 'NDVI_anomalia_pct_lag2',
    'NDVI_anomalia_pct_delta',
    'Gpp_anomalia_pct_lag1', 'Gpp_anomalia_pct_delta',
    'def_lag1', 'def_delta',
    'temp_aire_C_lag1', 'temp_aire_C_delta',
    'precipitation_lag1', 'precipitation_delta',
    'anio_relativo',
    'indice_estres_compuesto',
]

print(f'Nuevas features creadas: {len(new_features)}')
print('Nulos por feature nueva:')
print(df[new_features].isnull().sum().to_string())

## 3. Definición de conjuntos de features

In [ ]:
BASELINE = [
    'es_risaralda', 'precio_ico_usd_ton',
    'precipitation_annual_sum', 'temp_aire_C_annual_mean',
    'def_annual_mean', 'GDD_cafe_annual_mean',
    'NDVI_anomalia_pct_annual_mean',
    'precipitation_cosecha_sum', 'temp_aire_C_cosecha_mean',
    'NDVI_anomalia_pct_cosecha_mean',
]

SET_A = BASELINE + [
    'temp_x_def_annual', 'temp_x_def_cosecha',
    'precip_x_ndvi_anom_annual', 'precip_x_ndvi_anom_cosecha',
    'GDD_x_gpp_anom_cosecha',
    'balance_hidrico_annual', 'balance_hidrico_cosecha',
    'amplitud_termica_cosecha',
]

SET_B = BASELINE + [
    'gpp_ndvi_ratio_annual', 'gpp_ndvi_ratio_cosecha',
    'stress_hidrico_annual', 'stress_hidrico_cosecha',
    'balance_hidrico_annual', 'balance_hidrico_cosecha',
    'amplitud_termica_annual', 'amplitud_termica_cosecha',
    'indice_estres_compuesto',
]

SET_C = BASELINE + [
    'NDVI_anomalia_pct_lag1', 'NDVI_anomalia_pct_delta',
    'Gpp_anomalia_pct_lag1', 'Gpp_anomalia_pct_delta',
    'def_lag1', 'def_delta',
    'temp_aire_C_lag1', 'temp_aire_C_delta',
    'precipitation_lag1', 'precipitation_delta',
    'anio_relativo',
]

SET_D = BASELINE + [
    'temp_x_def_annual', 'temp_x_def_cosecha',
    'gpp_ndvi_ratio_cosecha',
    'stress_hidrico_annual', 'balance_hidrico_cosecha',
    'amplitud_termica_cosecha',
    'NDVI_anomalia_pct_lag1', 'NDVI_anomalia_pct_delta',
    'Gpp_anomalia_pct_lag1',
    'def_lag1', 'def_delta',
    'anio_relativo',
    'indice_estres_compuesto',
]

FEATURE_SETS = {
    'baseline':         BASELINE,
    'set_A_interacc':   SET_A,
    'set_B_compuestos': SET_B,
    'set_C_lags':       SET_C,
    'set_D_combinado':  SET_D,
}

for name, cols in FEATURE_SETS.items():
    print(f'{name:25s}: {len(cols)} variables')

fs_rows = []
for name, cols in FEATURE_SETS.items():
    fs_rows.append({'feature_set': name, 'n_vars': len(cols), 'variables': '|'.join(cols)})
pd.DataFrame(fs_rows).to_csv(OUT_DIR / 'feature_sets_utilizados.csv', index=False)
print('\nFeature sets guardados.')

## 4. Funciones auxiliares

In [ ]:
def add_indicator(df):
    """Añade es_risaralda si no existe."""
    if 'es_risaralda' not in df.columns:
        df = df.copy()
        df['es_risaralda'] = (df['departamento'] == 'Risaralda').astype(int)
    return df


def regression_metrics(y_true, y_pred, threshold=LOSS_THRESHOLD):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    ss_r = float(np.sum((y_true - y_pred) ** 2))
    ss_t = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2   = 1.0 - ss_r / (ss_t + 1e-12)
    dir_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    # clasificación derivada
    true_ev  = (y_true  <= threshold).astype(int)
    pred_ev  = (y_pred  <= threshold).astype(int)
    tp = int(np.sum((true_ev == 1) & (pred_ev == 1)))
    fp = int(np.sum((true_ev == 0) & (pred_ev == 1)))
    fn = int(np.sum((true_ev == 1) & (pred_ev == 0)))
    tn = int(np.sum((true_ev == 0) & (pred_ev == 0)))
    recall    = tp / (tp + fn + 1e-12)
    precision = tp / (tp + fp + 1e-12)
    f1        = 2 * precision * recall / (precision + recall + 1e-12)
    return dict(mae=mae, rmse=rmse, r2=r2, dir_acc=dir_acc,
                recall=recall, precision=precision, f1=f1,
                tp=tp, fp=fp, fn=fn, tn=tn)


def temporal_cv(df, features, target, model, min_train_years=5):
    """Validación cruzada temporal expandible por departamento."""
    years_train = sorted(df.loc[df['anio'] <= TRAIN_END_YEAR, 'anio'].unique())
    fold_results = []
    for split_year in years_train[min_train_years:]:
        tr = df[df['anio'] <= split_year].copy()
        va = df[(df['anio'] > split_year) & (df['anio'] <= TRAIN_END_YEAR)].copy()
        if len(va) == 0:
            continue
        Xtr, ytr = tr[features].fillna(0), tr[target]
        Xva, yva = va[features].fillna(0), va[target]
        if ytr.std() < 1e-6:
            continue
        try:
            model.fit(Xtr, ytr)
            preds = model.predict(Xva)
            m = regression_metrics(yva, preds)
            m['split_year'] = split_year
            fold_results.append(m)
        except Exception:
            pass
    if not fold_results:
        return {'cv_mae': np.nan, 'cv_rmse': np.nan, 'cv_r2': np.nan, 'cv_recall': np.nan}
    cv_df = pd.DataFrame(fold_results)
    return {
        'cv_mae':    cv_df['mae'].mean(),
        'cv_rmse':   cv_df['rmse'].mean(),
        'cv_r2':     cv_df['r2'].mean(),
        'cv_recall': cv_df['recall'].mean(),
        'n_folds':   len(cv_df),
    }


print('Funciones cargadas.')

## 5. Entrenamiento y evaluación con feature engineering

In [ ]:
df = add_indicator(df)

train_df = df[df['anio'] <= TRAIN_END_YEAR].copy()
test_df  = df[df['anio'] >  TRAIN_END_YEAR].copy()

print(f'Train: {len(train_df)} obs ({train_df.anio.min()}–{train_df.anio.max()})')
print(f'Test:  {len(test_df)}  obs ({test_df.anio.min()}–{test_df.anio.max()})')

# ── Grids de modelos ─────────────────────────────────────────────────────────
MODELS = {
    'Ridge': (
        lambda: Pipeline([('sc', StandardScaler()), ('m', Ridge())]),
        [{'m__alpha': [0.1, 1, 10, 50, 100]}]
    ),
    'HuberRegressor': (
        lambda: Pipeline([('sc', StandardScaler()), ('m', HuberRegressor(max_iter=300))]),
        [{'m__epsilon': [1.15, 1.35, 1.70], 'm__alpha': [0.0001, 0.001, 0.01]}]
    ),
    'RandomForest': (
        lambda: RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        [{'n_estimators': [300, 600], 'max_depth': [3, None], 'min_samples_leaf': [1, 3]}]
    ),
    'HistGradientBoosting': (
        lambda: HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        [{'max_depth': [2, 3], 'learning_rate': [0.03, 0.05],
          'max_iter': [200, 400], 'min_samples_leaf': [1, 3],
          'l2_regularization': [0.5, 1.0, 2.0]}]
    ),
    'ExtraTrees': (
        lambda: ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        [{'n_estimators': [300], 'max_depth': [3, None], 'min_samples_leaf': [2, 4]}]
    ),
}
if HAS_XGB:
    MODELS['XGBoost'] = (
        lambda: XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1),
        [{'n_estimators': [200, 400], 'max_depth': [2, 3],
          'learning_rate': [0.03, 0.05], 'reg_lambda': [1.0, 5.0],
          'subsample': [0.8, 1.0]}]
    )

all_results = []
total = sum(len(list(ParameterGrid(g[0]))) for _, (_, g) in MODELS.items()) * len(FEATURE_SETS)
done = 0

for fs_name, features in FEATURE_SETS.items():
    # features disponibles (puede haber lags con NaN en primeras filas)
    avail = [f for f in features if f in df.columns]
    Xtr = train_df[avail].fillna(0)
    ytr = train_df[TARGET]
    Xte = test_df[avail].fillna(0)
    yte = test_df[TARGET]

    for model_name, (model_fn, param_grids) in MODELS.items():
        for params in ParameterGrid(param_grids[0]):
            try:
                m = model_fn()
                m.set_params(**params)
                # CV
                cv = temporal_cv(train_df, avail, TARGET, model_fn())
                # Fit en todo train → test
                m.set_params(**params)
                m.fit(Xtr, ytr)
                preds_te = m.predict(Xte)
                te = regression_metrics(yte, preds_te)
                row = {
                    'model': model_name,
                    'feature_set': fs_name,
                    'n_features': len(avail),
                    **{f'param_{k}': v for k, v in params.items()},
                    'cv_mae': cv['cv_mae'],
                    'cv_rmse': cv['cv_rmse'],
                    'cv_r2': cv['cv_r2'],
                    'cv_recall': cv.get('cv_recall', np.nan),
                    'test_mae': te['mae'],
                    'test_rmse': te['rmse'],
                    'test_r2': te['r2'],
                    'test_recall': te['recall'],
                    'test_precision': te['precision'],
                    'test_f1': te['f1'],
                    'test_dir_acc': te['dir_acc'],
                    'tp': te['tp'], 'fp': te['fp'], 'fn': te['fn'], 'tn': te['tn'],
                }
                all_results.append(row)
            except Exception as e:
                pass
            done += 1
            if done % 50 == 0:
                print(f'  {done} combinaciones completadas...')

results_df = pd.DataFrame(all_results)
results_df.to_csv(OUT_DIR / 'resumen_todos_modelos.csv', index=False)
print(f'\nTotal combinaciones evaluadas: {len(results_df)}')
print(results_df[['model','feature_set','test_mae','test_r2','test_recall']]
      .sort_values('test_mae').head(15).to_string(index=False))

## 6. Selección del mejor modelo y comparación vs baseline

In [ ]:
# Filtrar no degenerados (std de predicciones > 1)
# Mejor modelo por test_mae
top = results_df.sort_values('test_mae').head(10).copy()
print('=== TOP 10 POR TEST-MAE ===')
print(top[['model','feature_set','test_mae','test_rmse','test_r2','test_recall','test_f1']]
      .to_string(index=False))

# Mejor por Recall de eventos
top_recall = results_df[results_df['test_recall'] > 0].sort_values(
    ['test_recall','test_mae'], ascending=[False, True]).head(10).copy()
print('\n=== TOP 10 PRIORIZANDO RECALL DE EVENTOS ===')
if len(top_recall) > 0:
    print(top_recall[['model','feature_set','test_mae','test_rmse','test_r2','test_recall','test_f1']]
          .to_string(index=False))
else:
    print('Ningún modelo detectó eventos con umbral fijo -15%.')
    print('(Se optimizará el umbral en la sección 7)')

# Comparación directa con baseline
baseline_row = results_df[
    (results_df['model'] == 'HistGradientBoosting') &
    (results_df['feature_set'] == 'baseline')
].sort_values('test_mae').head(1)
best_new = results_df.sort_values('test_mae').head(1)

print('\n=== COMPARACIÓN BASELINE vs MEJOR NUEVO ===')
if len(baseline_row) > 0:
    print(f"Baseline  (HGB-baseline):  MAE={baseline_row['test_mae'].values[0]:.2f}  R²={baseline_row['test_r2'].values[0]:.3f}")
print(f"Mejor nuevo ({best_new['model'].values[0]}-{best_new['feature_set'].values[0]}): "
      f"MAE={best_new['test_mae'].values[0]:.2f}  R²={best_new['test_r2'].values[0]:.3f}")

top.to_csv(OUT_DIR / 'top_modelos_avanzados.csv', index=False)
top_recall.to_csv(OUT_DIR / 'top_modelos_priorizando_recall.csv', index=False)

## 7. Optimización del umbral de clasificación

In [ ]:
# ── Entrenar el mejor modelo overall sobre train completo ────────────────────
best_row = results_df.sort_values('test_mae').iloc[0]
best_fs   = best_row['feature_set']
best_mname= best_row['model']
print(f'Mejor modelo: {best_mname} | features: {best_fs}')

# También evaluar el mejor modelo de regresión HGB en cada feature set
hgb_results = results_df[results_df['model'] == 'HistGradientBoosting'].sort_values('test_mae')

# Reentrenar el mejor configuración para obtener predicciones continuas
def get_best_model_preds(fs_name):
    avail = [f for f in FEATURE_SETS[fs_name] if f in df.columns]
    Xtr = train_df[avail].fillna(0)
    ytr = train_df[TARGET]
    Xte = test_df[avail].fillna(0)
    yte = test_df[TARGET]
    # Usar HGB con los mejores parámetros encontrados
    best_params_row = results_df[
        (results_df['model'] == 'HistGradientBoosting') &
        (results_df['feature_set'] == fs_name)
    ].sort_values('test_mae').head(1)
    if len(best_params_row) == 0:
        return None, None, None
    # extraer parámetros
    param_cols = [c for c in best_params_row.columns if c.startswith('param_')]
    params = {}
    for c in param_cols:
        k = c.replace('param_m__', '').replace('param_', '')
        v = best_params_row[c].values[0]
        if not pd.isna(v):
            params[k] = v
    m = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **params)
    m.fit(Xtr, ytr)
    preds = m.predict(Xte)
    return preds, yte.values, m

# Usar el mejor feature set
best_fs_for_thresh = hgb_results.iloc[0]['feature_set'] if len(hgb_results) > 0 else 'baseline'
preds_cont, y_true_te, best_model = get_best_model_preds(best_fs_for_thresh)

if preds_cont is None:
    # fallback a baseline
    preds_cont, y_true_te, best_model = get_best_model_preds('baseline')
    best_fs_for_thresh = 'baseline'

print(f'\nFeature set usado para optimización de umbral: {best_fs_for_thresh}')
print(f'Predicciones test: {preds_cont}')
print(f'Valores reales:    {y_true_te}')

In [ ]:
# ── ROC y precision-recall sobre TRAIN (k-fold temporal) ─────────────────────
# Para tener suficientes puntos para la curva, usamos predicciones OOF sobre train

avail_best = [f for f in FEATURE_SETS[best_fs_for_thresh] if f in df.columns]
oof_preds  = np.full(len(train_df), np.nan)
oof_years  = sorted(train_df['anio'].unique())

min_tr = 5
for i, yr in enumerate(oof_years[min_tr:], min_tr):
    tr_idx = train_df['anio'] < oof_years[i]
    va_idx = train_df['anio'] == oof_years[i]
    if tr_idx.sum() == 0 or va_idx.sum() == 0:
        continue
    Xtr_oof = train_df.loc[tr_idx, avail_best].fillna(0)
    ytr_oof = train_df.loc[tr_idx, TARGET]
    Xva_oof = train_df.loc[va_idx, avail_best].fillna(0)
    m_tmp = HistGradientBoostingRegressor(random_state=RANDOM_STATE,
                                           max_depth=2, learning_rate=0.03,
                                           l2_regularization=1.0, max_iter=200)
    m_tmp.fit(Xtr_oof, ytr_oof)
    oof_preds[va_idx.values] = m_tmp.predict(Xva_oof)

valid_mask = ~np.isnan(oof_preds)
y_oof = train_df.loc[valid_mask, TARGET].values
p_oof = oof_preds[valid_mask]

y_oof_bin = (y_oof <= LOSS_THRESHOLD).astype(int)

# Convertir predicción continua en score para ROC: negamos porque < -15 es evento
# score = -(pred) → mayor score = más negativo = más probabilidad de evento
score_oof = -p_oof

# Calcular curvas
if y_oof_bin.sum() >= 2:
    fpr, tpr, thresholds_roc = roc_curve(y_oof_bin, score_oof)
    roc_auc = auc(fpr, tpr)
    prec, rec, thresholds_pr = precision_recall_curve(y_oof_bin, score_oof)
    
    # Umbral óptimo: maximizar F1 con constraint Recall >= 0.70
    f1_scores = 2 * prec * rec / (prec + rec + 1e-12)
    recall_constraint = rec >= 0.70
    if recall_constraint.any():
        best_idx = np.argmax(np.where(recall_constraint, f1_scores, 0))
        opt_thresh_score = thresholds_pr[best_idx] if best_idx < len(thresholds_pr) else thresholds_pr[-1]
        # Convertir de vuelta a umbral en espacio de predicción: pred < -opt_thresh_score
        opt_pred_threshold = -opt_thresh_score
        opt_recall    = rec[best_idx]
        opt_precision = prec[best_idx]
        opt_f1        = f1_scores[best_idx]
    else:
        # Si no se alcanza recall 0.70, tomar el de mayor F1
        best_idx = np.argmax(f1_scores[:-1]) if len(f1_scores) > 1 else 0
        opt_pred_threshold = -thresholds_pr[best_idx] if best_idx < len(thresholds_pr) else LOSS_THRESHOLD
        opt_recall    = rec[best_idx]
        opt_precision = prec[best_idx]
        opt_f1        = f1_scores[best_idx]

    print(f'AUC-ROC (OOF train): {roc_auc:.3f}')
    print(f'Umbral óptimo de predicción: {opt_pred_threshold:.2f}%')
    print(f'→ Recall:    {opt_recall:.2f}')
    print(f'→ Precisión: {opt_precision:.2f}')
    print(f'→ F1:        {opt_f1:.2f}')
    
    # Guardar curvas
    pd.DataFrame({'fpr': fpr, 'tpr': tpr}).to_csv(OUT_DIR / 'curva_roc.csv', index=False)
    pd.DataFrame({'precision': prec, 'recall': rec}).to_csv(OUT_DIR / 'curva_pr.csv', index=False)
    pd.DataFrame({
        'umbral_prediccion': [opt_pred_threshold],
        'recall': [opt_recall],
        'precision': [opt_precision],
        'f1': [opt_f1],
        'auc_roc': [roc_auc],
    }).to_csv(OUT_DIR / 'umbral_optimo.csv', index=False)
    print('\nCurvas y umbral guardados.')
else:
    print('Pocos eventos en OOF para curva ROC confiable.')
    opt_pred_threshold = LOSS_THRESHOLD
    roc_auc = np.nan

In [ ]:
# ── Aplicar umbral óptimo al test set ────────────────────────────────────────
if preds_cont is not None:
    true_events_test = (y_true_te <= LOSS_THRESHOLD).astype(int)
    
    # Con umbral fijo -15
    pred_fixed  = (preds_cont <= LOSS_THRESHOLD).astype(int)
    # Con umbral óptimo
    pred_optim  = (preds_cont <= opt_pred_threshold).astype(int)
    
    def clf_metrics(y_t, y_p, label):
        tp = int(np.sum((y_t==1)&(y_p==1)))
        fp = int(np.sum((y_t==0)&(y_p==1)))
        fn = int(np.sum((y_t==1)&(y_p==0)))
        tn = int(np.sum((y_t==0)&(y_p==0)))
        rec = tp/(tp+fn+1e-9)
        pre = tp/(tp+fp+1e-9)
        f1  = 2*pre*rec/(pre+rec+1e-9)
        print(f'{label}: TP={tp} FP={fp} FN={fn} TN={tn} | Recall={rec:.2f} Prec={pre:.2f} F1={f1:.2f}')
        return dict(label=label,tp=tp,fp=fp,fn=fn,tn=tn,recall=rec,precision=pre,f1=f1)
    
    print('=== CLASIFICACIÓN EN TEST SET ===')
    r_fixed  = clf_metrics(true_events_test, pred_fixed,  f'Umbral fijo  ({LOSS_THRESHOLD}%)')
    r_optim  = clf_metrics(true_events_test, pred_optim,  f'Umbral óptimo ({opt_pred_threshold:.1f}%)')
    
    pd.DataFrame([r_fixed, r_optim]).to_csv(OUT_DIR / 'clasificacion_test_umbral_comparacion.csv', index=False)
    
    # Predicciones detalladas del test set
    pred_detail = test_df[['departamento','anio',TARGET,TARGET_CLF]].copy()
    pred_detail['pred_loss_pct'] = preds_cont
    pred_detail['pred_evento_umbral_fijo']  = pred_fixed
    pred_detail['pred_evento_umbral_optim'] = pred_optim
    pred_detail['error_abs'] = np.abs(y_true_te - preds_cont)
    pred_detail.to_csv(OUT_DIR / 'predicciones_test_con_umbrales.csv', index=False)
    print('\nPredicciones guardadas.')
    print(pred_detail.to_string(index=False))

## 8. Validación Leave-One-Department-Out (LOO)

In [ ]:
departments = df['departamento'].unique()
loo_results = []

for fs_name, features in FEATURE_SETS.items():
    avail = [f for f in features if f in df.columns]
    for test_dept in departments:
        train_loo = df[(df['departamento'] != test_dept) & (df['anio'] <= TRAIN_END_YEAR)].copy()
        test_loo  = df[(df['departamento'] == test_dept) & (df['anio'] >  TRAIN_END_YEAR)].copy()
        if len(train_loo) == 0 or len(test_loo) == 0:
            continue
        Xtr_l = train_loo[avail].fillna(0)
        ytr_l = train_loo[TARGET]
        Xte_l = test_loo[avail].fillna(0)
        yte_l = test_loo[TARGET]
        for model_name, (model_fn, param_grids) in [('HistGradientBoosting',
              MODELS['HistGradientBoosting'])]:
            for params in list(ParameterGrid(param_grids[0]))[:8]:  # primeras 8 configs
                try:
                    m = model_fn()
                    m.set_params(**params)
                    m.fit(Xtr_l, ytr_l)
                    preds_l = m.predict(Xte_l)
                    met = regression_metrics(yte_l, preds_l)
                    loo_results.append({
                        'test_department': test_dept,
                        'feature_set': fs_name,
                        'model': model_name,
                        **{f'param_{k}': v for k, v in params.items()},
                        'loo_mae': met['mae'], 'loo_rmse': met['rmse'],
                        'loo_r2': met['r2'], 'loo_recall': met['recall'],
                        'loo_precision': met['precision'], 'loo_f1': met['f1'],
                        'n_test': len(yte_l),
                    })
                except Exception:
                    pass

loo_df = pd.DataFrame(loo_results)
loo_df.to_csv(OUT_DIR / 'validacion_loo.csv', index=False)

if len(loo_df) > 0:
    print('=== RESULTADOS LOO POR DEPARTAMENTO ===')
    summary_loo = loo_df.groupby(['test_department','feature_set']).agg(
        loo_mae_min=('loo_mae','min'),
        loo_mae_mean=('loo_mae','mean'),
        loo_r2_max=('loo_r2','max'),
        loo_recall_max=('loo_recall','max'),
    ).reset_index()
    print(summary_loo.sort_values('loo_mae_min').to_string(index=False))
    summary_loo.to_csv(OUT_DIR / 'resumen_loo.csv', index=False)
else:
    print('No se generaron resultados LOO.')

## 9. Ensamble de Stacking

In [ ]:
# Stacking: HGB + RF + ExtraTrees → Ridge meta-learner
# Se usa el feature set con mejor desempeño individual

best_fs_stack = best_fs_for_thresh  # usar mismo que threshold optimization
avail_stack = [f for f in FEATURE_SETS[best_fs_stack] if f in df.columns]

Xtr_s = train_df[avail_stack].fillna(0)
ytr_s = train_df[TARGET]
Xte_s = test_df[avail_stack].fillna(0)
yte_s = test_df[TARGET]

estimators_stack = [
    ('hgb', HistGradientBoostingRegressor(
        random_state=RANDOM_STATE, max_depth=2,
        learning_rate=0.03, l2_regularization=1.0, max_iter=200)),
    ('rf',  RandomForestRegressor(
        random_state=RANDOM_STATE, n_estimators=300,
        max_depth=3, min_samples_leaf=3, n_jobs=-1)),
    ('et',  ExtraTreesRegressor(
        random_state=RANDOM_STATE, n_estimators=300,
        max_depth=3, min_samples_leaf=3, n_jobs=-1)),
]
if HAS_XGB:
    estimators_stack.append((
        'xgb', XGBRegressor(
            random_state=RANDOM_STATE, n_estimators=200, max_depth=2,
            learning_rate=0.05, reg_lambda=1.0, subsample=0.8, verbosity=0)
    ))

stack_model = StackingRegressor(
    estimators=estimators_stack,
    final_estimator=Ridge(alpha=10.0),
    cv=5,
    passthrough=False,
)

try:
    stack_model.fit(Xtr_s, ytr_s)
    preds_stack = stack_model.predict(Xte_s)
    met_stack = regression_metrics(yte_s, preds_stack)
    print(f'=== STACKING ({best_fs_stack}) ===')
    print(f"MAE:     {met_stack['mae']:.2f} pp")
    print(f"RMSE:    {met_stack['rmse']:.2f} pp")
    print(f"R²:      {met_stack['r2']:.3f}")
    print(f"Recall:  {met_stack['recall']:.2f}")
    print(f"F1:      {met_stack['f1']:.2f}")
    print(f"Dir.Acc: {met_stack['dir_acc']:.2f}")
    
    # Aplicar umbral óptimo
    pred_stack_optim = (preds_stack <= opt_pred_threshold).astype(int)
    true_ev = (yte_s.values <= LOSS_THRESHOLD).astype(int)
    tp=int(np.sum((true_ev==1)&(pred_stack_optim==1)))
    fp=int(np.sum((true_ev==0)&(pred_stack_optim==1)))
    fn=int(np.sum((true_ev==1)&(pred_stack_optim==0)))
    tn=int(np.sum((true_ev==0)&(pred_stack_optim==0)))
    rec=tp/(tp+fn+1e-9); pre=tp/(tp+fp+1e-9); f1=2*pre*rec/(pre+rec+1e-9)
    print(f'\nCon umbral óptimo ({opt_pred_threshold:.1f}%): TP={tp} FP={fp} FN={fn} TN={tn} | Recall={rec:.2f} F1={f1:.2f}')
    
    stacking_result = pd.DataFrame([{
        'model': 'StackingRegressor',
        'feature_set': best_fs_stack,
        'base_models': str([n for n,_ in estimators_stack]),
        'meta_model': 'Ridge(alpha=10)',
        'test_mae': met_stack['mae'], 'test_rmse': met_stack['rmse'],
        'test_r2': met_stack['r2'], 'test_recall_fixed': met_stack['recall'],
        'test_recall_optim': rec, 'test_f1_optim': f1,
    }])
    stacking_result.to_csv(OUT_DIR / 'resultado_stacking.csv', index=False)
except Exception as e:
    print(f'Error en stacking: {e}')

## 10. Calibración del trigger del seguro (basis risk)

In [ ]:
# ── Simular pagos del seguro bajo diferentes reglas de trigger ───────────────
# Regla del seguro: si índice_predicho <= trigger_threshold → pagar (1-SLR × pérdida)
# Basis risk = |pérdida real - pago simulado|

if preds_cont is not None and len(y_true_te) > 0:
    # Reconstruir con TODOS los datos disponibles (train + test) para tener suficientes puntos
    avail_all = [f for f in FEATURE_SETS[best_fs_for_thresh] if f in df.columns]
    has_agronet = df['tiene_observacion_agronet'] == True if 'tiene_observacion_agronet' in df.columns \
                  else df['anio'].between(2007, 2024)
    df_eval = df[has_agronet].copy()
    
    Xall = df_eval[avail_all].fillna(0)
    # Usar modelo entrenado en train para predecir todo el período
    best_model.fit(
        train_df[avail_all].fillna(0),
        train_df[TARGET]
    )
    df_eval = df_eval.copy()
    df_eval['pred_loss_pct'] = best_model.predict(Xall)
    df_eval['real_loss_pct'] = df_eval[TARGET]
    df_eval['evento_real'] = (df_eval[TARGET] <= LOSS_THRESHOLD).astype(int)
    
    # Simular bajo diferentes umbrales de trigger
    trigger_thresholds = np.arange(-25, 5, 1.0)   # barrido de -25% a +5%
    slr = 1.0  # stop-loss ratio: pagar 100% de la pérdida estimada
    
    trigger_results = []
    for tt in trigger_thresholds:
        df_eval['trigger_activado'] = (df_eval['pred_loss_pct'] <= tt).astype(int)
        # Pago = pérdida predicha cuando trigger activo, 0 cuando no
        df_eval['pago_seguro_pct'] = np.where(
            df_eval['trigger_activado'] == 1,
            np.abs(df_eval['pred_loss_pct']),   # pago proporcional a pérdida predicha
            0.0
        )
        # Basis risk por observación
        df_eval['perdida_no_cubierta'] = np.maximum(0,
            np.abs(df_eval['real_loss_pct']) * df_eval['evento_real'] - df_eval['pago_seguro_pct'])
        df_eval['basis_risk'] = np.abs(
            df_eval['real_loss_pct'] * df_eval['evento_real'] - df_eval['pago_seguro_pct'])
        
        # Métricas de clasificación del trigger
        tp_ = int(np.sum((df_eval['evento_real']==1) & (df_eval['trigger_activado']==1)))
        fp_ = int(np.sum((df_eval['evento_real']==0) & (df_eval['trigger_activado']==1)))
        fn_ = int(np.sum((df_eval['evento_real']==1) & (df_eval['trigger_activado']==0)))
        tn_ = int(np.sum((df_eval['evento_real']==0) & (df_eval['trigger_activado']==0)))
        rec_  = tp_/(tp_+fn_+1e-9)
        pre_  = tp_/(tp_+fp_+1e-9)
        f1_   = 2*pre_*rec_/(pre_+rec_+1e-9)
        
        trigger_results.append({
            'trigger_threshold_pct': tt,
            'n_activaciones': df_eval['trigger_activado'].sum(),
            'recall_eventos': rec_,
            'precision_eventos': pre_,
            'f1_eventos': f1_,
            'basis_risk_media': df_eval['basis_risk'].mean(),
            'basis_risk_std': df_eval['basis_risk'].std(),
            'perdida_no_cubierta_media': df_eval['perdida_no_cubierta'].mean(),
            'tp': tp_, 'fp': fp_, 'fn': fn_, 'tn': tn_,
        })
    
    trig_df = pd.DataFrame(trigger_results)
    trig_df.to_csv(OUT_DIR / 'calibracion_trigger_basis_risk.csv', index=False)
    
    # Encontrar umbral óptimo: recall >= 0.70 y mínimo basis risk
    viable = trig_df[trig_df['recall_eventos'] >= 0.70]
    if len(viable) > 0:
        best_trigger = viable.sort_values('basis_risk_media').iloc[0]
        print(f'=== TRIGGER ÓPTIMO (Recall ≥ 0.70) ===')
        print(f"Umbral: {best_trigger['trigger_threshold_pct']:.1f}%")
        print(f"Recall: {best_trigger['recall_eventos']:.2f}")
        print(f"Precisión: {best_trigger['precision_eventos']:.2f}")
        print(f"F1: {best_trigger['f1_eventos']:.2f}")
        print(f"Basis Risk Medio: {best_trigger['basis_risk_media']:.2f} pp")
        print(f"Pérdida No Cubierta Media: {best_trigger['perdida_no_cubierta_media']:.2f} pp")
        pd.DataFrame([best_trigger]).to_csv(OUT_DIR / 'trigger_optimo.csv', index=False)
    else:
        print('Ningún umbral alcanza Recall ≥ 0.70 en el conjunto histórico completo.')
        # Reportar el de mayor recall
        best_trigger = trig_df.sort_values('recall_eventos', ascending=False).iloc[0]
        print(f"Mejor recall alcanzado: {best_trigger['recall_eventos']:.2f} con umbral {best_trigger['trigger_threshold_pct']:.1f}%")
    
    print('\nCurva de basis risk guardada.')
    print(trig_df[['trigger_threshold_pct','recall_eventos','basis_risk_media']]
          .to_string(index=False))

## 11. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Resultados – Notebook 11: Feature Engineering y Modelado Avanzado',
             fontsize=14, fontweight='bold', y=1.01)

# ── Panel 1: Comparación MAE por feature set ─────────────────────────────────
ax1 = axes[0, 0]
comp = results_df.groupby('feature_set')['test_mae'].min().reset_index()
colors_bar = ['#2E4057' if 'baseline' in x else '#1A6E9C' for x in comp['feature_set']]
ax1.bar(comp['feature_set'], comp['test_mae'], color=colors_bar, edgecolor='white')
ax1.set_title('Mejor MAE (test) por Feature Set', fontweight='bold')
ax1.set_ylabel('MAE (pp)')
ax1.set_ylim(0, max(comp['test_mae']) * 1.3)
ax1.tick_params(axis='x', rotation=35)
for i, (_, row) in enumerate(comp.iterrows()):
    ax1.text(i, row['test_mae'] + 0.2, f"{row['test_mae']:.1f}", ha='center', fontsize=9)

# ── Panel 2: Curva ROC ───────────────────────────────────────────────────────
ax2 = axes[0, 1]
try:
    roc_data = pd.read_csv(OUT_DIR / 'curva_roc.csv')
    ax2.plot(roc_data['fpr'], roc_data['tpr'], color='#1A6E9C', lw=2,
             label=f'ROC (AUC={roc_auc:.2f})')
    ax2.plot([0,1],[0,1],'--',color='gray', alpha=0.6, label='Aleatorio')
    ax2.set_xlabel('Tasa de Falsos Positivos'); ax2.set_ylabel('Recall (TPR)')
    ax2.set_title('Curva ROC – Detección de Eventos (OOF Train)', fontweight='bold')
    ax2.legend(); ax2.set_xlim([-0.02,1.02]); ax2.set_ylim([-0.02,1.05])
except:
    ax2.text(0.5, 0.5, 'ROC no disponible', ha='center', va='center')
    ax2.set_title('Curva ROC', fontweight='bold')

# ── Panel 3: Precisión-Recall ────────────────────────────────────────────────
ax3 = axes[0, 2]
try:
    pr_data = pd.read_csv(OUT_DIR / 'curva_pr.csv')
    ax3.plot(pr_data['recall'], pr_data['precision'], color='#E8745A', lw=2)
    ax3.axvline(x=0.70, linestyle='--', color='#2E4057', alpha=0.7, label='Recall = 0.70')
    if 'opt_recall' in dir():
        ax3.scatter([opt_recall], [opt_precision], color='red', zorder=5, s=80,
                    label=f'Óptimo (F1={opt_f1:.2f})')
    ax3.set_xlabel('Recall'); ax3.set_ylabel('Precisión')
    ax3.set_title('Curva Precisión-Recall', fontweight='bold')
    ax3.legend(); ax3.set_xlim([-0.02,1.05]); ax3.set_ylim([-0.02,1.05])
except:
    ax3.text(0.5, 0.5, 'PR no disponible', ha='center', va='center')
    ax3.set_title('Curva P-R', fontweight='bold')

# ── Panel 4: Real vs Predicho (test) ─────────────────────────────────────────
ax4 = axes[1, 0]
if preds_cont is not None:
    colors_dept = ['#2E4057' if d == 'Risaralda' else '#1A6E9C'
                   for d in test_df['departamento']]
    ax4.scatter(y_true_te, preds_cont, c=colors_dept, s=80, edgecolors='white', zorder=3)
    lims = [min(y_true_te.min(), preds_cont.min()) - 3,
            max(y_true_te.max(), preds_cont.max()) + 3]
    ax4.plot(lims, lims, '--', color='gray', alpha=0.7, label='Predicción perfecta')
    ax4.axhline(y=LOSS_THRESHOLD, color='red', linestyle=':', alpha=0.6, label=f'Umbral {LOSS_THRESHOLD}%')
    ax4.axvline(x=LOSS_THRESHOLD, color='red', linestyle=':', alpha=0.6)
    ax4.set_xlabel('Pérdida Real (%)'); ax4.set_ylabel('Pérdida Predicha (%)')
    ax4.set_title('Real vs Predicho – Test 2021–2024', fontweight='bold')
    patch1 = mpatches.Patch(color='#2E4057', label='Risaralda')
    patch2 = mpatches.Patch(color='#1A6E9C', label='Cundinamarca')
    ax4.legend(handles=[patch1, patch2], loc='upper left')
    for i, row in enumerate(test_df.itertuples()):
        ax4.annotate(f"{row.anio}", (y_true_te[i], preds_cont[i]),
                     textcoords='offset points', xytext=(4, 4), fontsize=7)
    mae_val = np.mean(np.abs(y_true_te - preds_cont))
    ax4.set_title(f'Real vs Predicho – Test 2021–2024 (MAE={mae_val:.1f}pp)', fontweight='bold')

# ── Panel 5: Basis Risk vs Umbral del Trigger ────────────────────────────────
ax5 = axes[1, 1]
try:
    trig_plot = pd.read_csv(OUT_DIR / 'calibracion_trigger_basis_risk.csv')
    ax5_twin = ax5.twinx()
    ax5.plot(trig_plot['trigger_threshold_pct'], trig_plot['basis_risk_media'],
             color='#E8745A', lw=2, label='Basis Risk Medio')
    ax5_twin.plot(trig_plot['trigger_threshold_pct'], trig_plot['recall_eventos'],
                  color='#2E4057', lw=2, linestyle='--', label='Recall Eventos')
    ax5.axvline(x=LOSS_THRESHOLD, color='gray', linestyle=':', alpha=0.7, label='Umbral -15%')
    try:
        opt_tt = best_trigger['trigger_threshold_pct']
        ax5.axvline(x=opt_tt, color='green', linestyle='--', alpha=0.8, label=f'Óptimo ({opt_tt:.0f}%)')
    except:
        pass
    ax5.set_xlabel('Umbral del Trigger (%)')
    ax5.set_ylabel('Basis Risk Medio (pp)', color='#E8745A')
    ax5_twin.set_ylabel('Recall de Eventos', color='#2E4057')
    ax5.set_title('Calibración del Trigger – Basis Risk vs Recall', fontweight='bold')
    lines1, labels1 = ax5.get_legend_handles_labels()
    lines2, labels2 = ax5_twin.get_legend_handles_labels()
    ax5.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
except Exception as e:
    ax5.text(0.5, 0.5, 'Datos de trigger no disponibles', ha='center', va='center')
    ax5.set_title('Calibración del Trigger', fontweight='bold')

# ── Panel 6: Resumen comparativo final ───────────────────────────────────────
ax6 = axes[1, 2]
ax6.axis('off')
best_overall = results_df.sort_values('test_mae').iloc[0]

# Baseline reference
b_row = results_df[(results_df['model']=='HistGradientBoosting') &
                   (results_df['feature_set']=='baseline')].sort_values('test_mae')
b_mae  = b_row['test_mae'].values[0]  if len(b_row) else np.nan
b_r2   = b_row['test_r2'].values[0]   if len(b_row) else np.nan
b_rec  = b_row['test_recall'].values[0] if len(b_row) else np.nan

summary_text = (
    "RESUMEN COMPARATIVO\n"
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    f"BASELINE (HGB):\n"
    f"  MAE  = {b_mae:.2f} pp\n"
    f"  R²   = {b_r2:.3f}\n"
    f"  Recall = {b_rec:.2f}\n\n"
    f"MEJOR NUEVO ({best_overall['model'][:12]}):\n"
    f"  FS   = {best_overall['feature_set']}\n"
    f"  MAE  = {best_overall['test_mae']:.2f} pp\n"
    f"  R²   = {best_overall['test_r2']:.3f}\n"
    f"  Recall = {best_overall['test_recall']:.2f}\n\n"
    f"UMBRAL ÓPTIMO:\n"
    f"  Thresh = {opt_pred_threshold:.1f}%\n"
    f"  AUC-ROC = {roc_auc:.3f}\n"
)
ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#EEF2F7', alpha=0.8))
ax6.set_title('Resumen de Resultados', fontweight='bold')

plt.tight_layout()
fig_path = OUT_DIR / 'dashboard_resultados_nb11.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Dashboard guardado: {fig_path}')

## 12. Resumen consolidado

In [ ]:
print('=' * 65)
print('NOTEBOOK 11 – RESUMEN DE RESULTADOS FINALES')
print('=' * 65)

print('\n📊 MEJORES MODELOS POR TEST-MAE:')
print(results_df.sort_values('test_mae')
      [['model','feature_set','test_mae','test_rmse','test_r2','test_recall']]
      .head(5).to_string(index=False))

if len(top_recall) > 0:
    print('\n🎯 MEJORES MODELOS POR RECALL DE EVENTOS (umbral fijo):')
    print(top_recall[['model','feature_set','test_mae','test_recall','test_f1']]
          .head(5).to_string(index=False))

print(f'\n🔢 UMBRAL ÓPTIMO DE CLASIFICACIÓN: {opt_pred_threshold:.1f}%')
print(f'   AUC-ROC (OOF): {roc_auc:.3f}')
print(f'   Recall esperado: {opt_recall:.2f}')
print(f'   F1 esperado: {opt_f1:.2f}')

if len(loo_df) > 0:
    print('\n🔄 VALIDACIÓN LOO – Mejor por departamento:')
    best_loo = loo_df.sort_values('loo_mae').groupby('test_department').head(1)
    print(best_loo[['test_department','feature_set','loo_mae','loo_r2','loo_recall']]
          .to_string(index=False))

print(f'\n📁 Resultados en: {OUT_DIR}')
files = list(OUT_DIR.glob('*.csv')) + list(OUT_DIR.glob('*.png'))
for f in sorted(files):
    print(f'  {f.name}')

print('\n✅ Notebook 11 completado.')